In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

from torchvision import datasets
from torchvision.transforms import ToTensor

In [ ]:
train_data = datasets.MNIST(
    root = 'mnist_data',
    train = True,
    transform = ToTensor(),
    download = True,
)
test_data = datasets.MNIST(
    root = 'mnist_data',
    train = False,
    transform = ToTensor()
)

X_train = train_data.data.view((-1, 28*28)).float() / 255.0
X_test = test_data.data.view((-1, 28*28)).float() / 255.0

y_train = train_data.targets
y_test = test_data.targets

In [ ]:
torch.manual_seed(12345)

model = nn.Sequential(
    nn.Linear(784, 256),   # Hidden layer: 784 inputs -> 256 outputs (with bias)
    nn.ReLU(),             # Activation function
    nn.Linear(256, 10),    # Output layer: 256 inputs -> 10 outputs (with bias)
)

# CrossEntropyLoss combines log-softmax and negative log-likelihood,
# so the model outputs raw scores (logits), not probabilities.
loss_fn = nn.CrossEntropyLoss()

# Stochastic Gradient Descent, same as the manual: p.data += -lr * p.grad
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [ ]:
m = 64  # Batch size
g = torch.Generator().manual_seed(12345)

In [ ]:
# Training
n_epochs = 213
for epoch in range(n_epochs):

    # Shuffle the training set at the start of each epoch
    perm = torch.randperm(len(X_train), generator=g)

    for i in range(len(X_train) // m):
        ix = perm[i*m : (i+1)*m]
        X = X_train[ix]
        y = y_train[ix]

        # Forward pass
        logits = model(X)
        loss = loss_fn(logits, y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if epoch % 20 == 0:
        print(f"epoch = {epoch:3},  loss = {loss:.4}")

In [ ]:
start = 500
for k in range(5):
    X = X_test[start+k]

    # Get probabilities using softmax on the model's raw output
    probs = torch.softmax(model(X), dim=0)

    plt.figure(figsize=(10, 4))
    plt.subplot(1,2,1)
    plt.imshow(X.view(28, 28), cmap='gray')
    ind = np.arange(0, 10)
    plt.subplot(1,2,2)
    plt.bar(ind, probs.detach().numpy().flatten())
    plt.xticks(range(10))